# Neural Collaborative Filtering (NCF) for Recommendation

## 1. Imports

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import random

from helpers.data_loaders import load_movielens_data, load_steam_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


## 2. Data Loading and Preparation

In [2]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

# For NCF, we'll treat any interaction as a positive signal.
# Movielens: rating >= 4.0 is a positive interaction
ratings_df_filtered = ratings_df[ratings_df['rating'] >= 4.0].copy()
movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df_filtered['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df_filtered['movieId'].astype(str)
})

# Steam: any review is a positive interaction
steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str)
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

unique_users = all_interactions['user_id'].unique()
unique_items = all_interactions['item_id'].unique()

user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}

num_users = len(user_map)
num_items = len(item_map)

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

all_interactions['user_idx'] = all_interactions['user_id'].map(user_map)
all_interactions['item_idx'] = all_interactions['item_id'].map(item_map)


Total unique interactions: 12497401
Number of users: 184419
Number of items: 43660


## 3. NCF Model Definition

In [3]:
from src.models.ncf import NCFModel as NCF

## 4. Training Setup

In [4]:
from src.models.ncf import NCFDataset

embedding_dim_gmf = 32
embedding_dim_mlp = 32
mlp_layers = [64, 32, 16]
batch_size = 1024
learning_rate = 1e-3
epochs = 1
num_neg_samples = 4

train_dataset = NCFDataset(all_interactions, num_items, num_neg_samples)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = NCF(num_users, num_items, embedding_dim_gmf, embedding_dim_mlp, mlp_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()


## 5. Training Loop

In [5]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for user_batch, item_batch, label_batch in progress_bar:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        label_batch = label_batch.float().to(device)
        
        optimizer.zero_grad()
        
        predictions = model(user_batch, item_batch)
        
        loss = loss_fn(predictions, label_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")


Epoch 1/1:   0%|          | 0/61023 [00:00<?, ?it/s]

Epoch 1/1, Average Loss: 0.1257


## 6. Making Recommendations

In [7]:
def get_recommendations_ncf(user_id_str, model, top_k=10):
    model.eval()
    
    if user_id_str not in user_map:
        print(f"User '{user_id_str}' not found.")
        return
    user_idx = user_map[user_id_str]
    
    with torch.no_grad():
        user_idx_tensor = torch.LongTensor([user_idx]).to(device)
        item_indices_tensor = torch.LongTensor(list(item_map.values())).to(device)
        
        # Predict scores for all items for the given user
        scores = model(user_idx_tensor.repeat(num_items), item_indices_tensor)
        
        # Remove items the user has already interacted with
        liked_items_indices = all_interactions[all_interactions['user_id'] == user_id_str]['item_idx'].values
        scores[liked_items_indices] = -np.inf # Set to a very low value
        
        top_k_scores, top_k_indices = torch.topk(scores, k=top_k)
        
        inv_item_map = {i: item for item, i in item_map.items()}
        
        print(f"Top {top_k} recommendations for user '{user_id_str}':")
        for i, score in zip(top_k_indices.cpu().numpy(), top_k_scores.cpu().numpy()):
            item_id_str = inv_item_map[i]
            if 'steam' in item_id_str:
                item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item_id_str.split('_')[-1])]
                if not item_info.empty:
                    print(f"  - [steam] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")
            else:
                item_info = movies_df.loc[movies_df['movieId'] == int(item_id_str.split('_')[-1])]
                if not item_info.empty:
                    print(f"  - [movie] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")

sample_user_id = 'steam_user_LydiaMorley'
print(f"Items liked by the user ({sample_user_id}):")

liked_items = all_interactions[all_interactions['user_id'] == sample_user_id]['item_id']
for item in liked_items:
    if 'steam' in item:
        item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item.split('_')[-1])]
        if not item_info.empty:
            print(f"  - {item}, Title: {item_info['title'].values[0]}")
    else:
        item_info = movies_df.loc[movies_df['movieId'] == int(item.split('_')[-1])]
        if not item_info.empty:
            print(f"  - {item}, Title: {item_info['title'].values[0]}")

get_recommendations_ncf(sample_user_id, model)


Items liked by the user (steam_user_LydiaMorley):
  - steam_item_273110, Title: Counter-Strike Nexon: Zombies
  - steam_item_730, Title: Counter-Strike: Global Offensive
  - steam_item_440, Title: Team Fortress 2
Top 10 recommendations for user 'steam_user_LydiaMorley':
  - [steam] Item: steam_item_4000, Title: Garry's Mod , Score: 8.5840
  - [steam] Item: steam_item_49520, Title: Borderlands 2 , Score: 7.3010
  - [steam] Item: steam_item_8930, Title: Sid Meier's Civilization® V , Score: 7.0602
  - [steam] Item: steam_item_304930, Title: Unturned , Score: 6.8223
  - [steam] Item: steam_item_252490, Title: Rust , Score: 6.7258
  - [steam] Item: steam_item_105600, Title: Terraria , Score: 6.5256
  - [steam] Item: steam_item_218230, Title: PlanetSide 2 , Score: 6.4612
  - [steam] Item: steam_item_224260, Title: No More Room in Hell , Score: 6.4310
  - [steam] Item: steam_item_550, Title: Left 4 Dead 2 , Score: 6.0965
  - [steam] Item: steam_item_107410, Title: Arma 3 , Score: 5.9149
